# B2-020-language-transformers — Practice p21 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** nlp-fine-tuning-protocol

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Pinned reproducibility protocol

Load the sole encoder state and literal classification rows from `data/language_fixture.py`; seed the classifier with `20260812`.  AdamW uses `lr=0.03`, `weight_decay=0`, betas `(0.9,0.999)`, eps `1e-8`, and stored ascending full batches.  Run 20 + 20 full-batch updates: first classifier-only with every encoder parameter frozen, then encoder plus classifier after deliberate unfreezing.

## Solution

Stage 1 deliberately excludes frozen encoder parameters. Stage 2 deliberately unfreezes them and builds a new optimizer whose parameters are the exact returned encoder and classifier objects.

In [ ]:
import importlib.util

def load_literal_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, relative_path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)

class TinyEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(12, 8, padding_idx=0)
        self.position_embedding = nn.Embedding(8, 8)
        self.norm1 = nn.LayerNorm(8, eps=1e-5)
        self.attention = nn.MultiheadAttention(8, 2, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(8, eps=1e-5)
        self.ff1 = nn.Linear(8, 16)
        self.ff2 = nn.Linear(16, 8)

    def forward(self, token_ids, *, mask_mode):
        length = token_ids.shape[1]
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        normalized = self.norm1(x)
        attention_mask = None
        if mask_mode == "causal":
            attention_mask = torch.triu(
                torch.ones(length, length, dtype=torch.bool, device=token_ids.device),
                diagonal=1,
            )
        elif mask_mode != "bidirectional":
            raise ValueError(f"unknown mask mode: {mask_mode}")
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            key_padding_mask=token_ids.eq(0),
            need_weights=False,
        )
        x = x + attended
        return x + self.ff2(F.gelu(self.ff1(self.norm2(x))))

fixture = load_literal_module("language_fixture_p21", "../data/language_fixture.py")
state = load_literal_module("tiny_encoder_state_p21", "../data/tiny_encoder_state.py")
EXPECTED_STATE_HASH = "e26bfa7738beb32e415ff88c7da73fc95dfcdcdcc6bab3d19d7c3019b0319150"

def configure_frozen_stage_optimizer(encoder, classifier):
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    return torch.optim.AdamW(classifier.parameters(), lr=0.03, weight_decay=0, betas=(0.9,0.999), eps=1e-8)

def pool(encoder, rows):
    hidden = encoder(rows, mask_mode="bidirectional")
    valid = rows.ne(0)
    return (hidden * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True)

computed_state_hash = state.canonical_encoder_state_hash(state.ENCODER_TENSORS)
assert state.ENCODER_STATE_HASH == EXPECTED_STATE_HASH == computed_state_hash
encoder = TinyEncoder()
expected_encoder_state = {name: torch.tensor(value, dtype=torch.float32) for name, value in state.ENCODER_TENSORS.items()}
encoder.load_state_dict(expected_encoder_state, strict=True)
loaded_state_verified = all(torch.equal(value, expected_encoder_state[name]) for name, value in encoder.state_dict().items())
torch.manual_seed(20260812)
classifier = nn.Linear(8, 2)
rows = torch.tensor(fixture.INTENT_INPUT_IDS, dtype=torch.int64)[fixture.TRAIN_SPLIT_IDS]
labels = torch.tensor(fixture.INTENT_LABELS, dtype=torch.int64)[fixture.TRAIN_SPLIT_IDS]
encoder_before = {name: value.detach().clone() for name, value in encoder.state_dict().items()}
head_before = {name: value.detach().clone() for name, value in classifier.state_dict().items()}
stage1_optimizer = configure_frozen_stage_optimizer(encoder, classifier)
stage1_ids = {id(parameter) for group in stage1_optimizer.param_groups for parameter in group["params"]}
for _ in range(20):
    stage1_optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(classifier(pool(encoder, rows)), labels)
    loss.backward(); stage1_optimizer.step()
encoder_after_stage1 = {name: value.detach().clone() for name, value in encoder.state_dict().items()}
head_after_stage1 = {name: value.detach().clone() for name, value in classifier.state_dict().items()}
for parameter in encoder.parameters():
    parameter.requires_grad_(True)
stage2_optimizer = torch.optim.AdamW([*encoder.parameters(), *classifier.parameters()], lr=0.03, weight_decay=0, betas=(0.9,0.999), eps=1e-8)
stage2_ids = {id(parameter) for group in stage2_optimizer.param_groups for parameter in group["params"]}
for _ in range(20):
    stage2_optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(classifier(pool(encoder, rows)), labels)
    loss.backward(); stage2_optimizer.step()
encoder_after_stage2 = encoder.state_dict()
head_after_stage2 = classifier.state_dict()

### Answer check

In [ ]:
assert loaded_state_verified
assert stage1_ids == {id(parameter) for parameter in classifier.parameters()}
assert all(torch.allclose(encoder_before[name], encoder_after_stage1[name], atol=1e-7, rtol=1e-7) for name in encoder_before)
assert any(not torch.allclose(head_before[name], head_after_stage1[name], atol=1e-7, rtol=1e-7) for name in head_before)
assert stage2_ids == {id(parameter) for parameter in [*encoder.parameters(), *classifier.parameters()]}
assert any(not torch.allclose(encoder_after_stage1[name], encoder_after_stage2[name], atol=1e-7, rtol=1e-7) for name in encoder_after_stage1)
assert any(not torch.allclose(head_after_stage1[name], head_after_stage2[name], atol=1e-7, rtol=1e-7) for name in head_after_stage1)